In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd

titanic = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
!pip install duckdb

In [4]:
import duckdb

In [5]:
duckdb.sql("""
SELECT *
FROM titanic
LIMIT 5;
""").df()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


In [6]:
duckdb.sql("""
SELECT COUNT(*) AS total_passengers
FROM titanic;
""").df()

,total_passengers
0,891


Insight: Counts the total number of passengers.

In [7]:
duckdb.sql("""
SELECT COUNT(*) AS survived
FROM titanic
WHERE Survived = 1;
""").df()

,survived
0,342


Insight: Counts the total number of survivors.

In [8]:
duckdb.sql("""
SELECT
    Survived,
    COUNT(*) AS passengers
FROM titanic
GROUP BY Survived;
""").df()

,Survived,passengers
0,0,549
1,1,342


In [9]:
duckdb.sql("""
SELECT
    Pclass,
    COUNT(*) AS total_passengers
FROM titanic
GROUP BY Pclass
ORDER BY Pclass;
""").df()

,Pclass,total_passengers
0,1,216
1,2,184
2,3,491


Insight: Which passenger class had the most passengers?

In [10]:
duckdb.sql("""
SELECT
    Pclass,
    COUNT(*) AS survivors
FROM titanic
WHERE Survived = 1
GROUP BY Pclass
ORDER BY Pclass;
""").df()

,Pclass,survivors
0,1,136
1,2,87
2,3,119


Insight: Which class had the highest number of survivors?

In [11]:
duckdb.sql("""
SELECT
    Sex,
    COUNT(*) AS total_passengers
FROM titanic
GROUP BY Sex;
""").df()

,Sex,total_passengers
0,female,314
1,male,577


Insight: Number of male and female passengers

In [12]:
duckdb.sql("""
SELECT
    Sex,
    COUNT(*) AS survivors
FROM titanic
WHERE Survived = 1
GROUP BY Sex;
""").df()

,Sex,survivors
0,male,109
1,female,233


Insight: Number of male and female survivors 

In [13]:
duckdb.sql("""
SELECT
    ROUND(AVG(Age),2) AS average_age
FROM titanic;
""").df()

,average_age
0,29.7


Insight: Average age of passengers 

In [14]:
duckdb.sql("""
SELECT
    MIN(Age) AS youngest,
    MAX(Age) AS oldest
FROM titanic;
""").df()

,youngest,oldest
0,0.42,80.0


Insight: Youngest and oldest passenger 

In [15]:
duckdb.sql("""
SELECT
    Pclass,
    ROUND(AVG(Fare),2) AS average_fare
FROM titanic
GROUP BY Pclass
ORDER BY Pclass;
""").df()

,Pclass,average_fare
0,1,84.15
1,2,20.66
2,3,13.68


Insight: Average fare by passenger class 

In [16]:
duckdb.sql("""
SELECT
    Name,
    Fare
FROM titanic
ORDER BY Fare DESC
LIMIT 10;
""").df()

,Name,Fare
0,"Lesurer, Mr. Gustave J",512.3292
1,"Ward, Miss. Anna",512.3292
2,"Cardeza, Mr. Thomas Drake Martinez",512.3292
3,"Fortune, Miss. Mabel Helen",263.0000
4,"Fortune, Mr. Charles Alexander",263.0000
5,"Fortune, Mr. Mark",263.0000
6,"Fortune, Miss. Alice Elizabeth",263.0000
7,"Ryerson, Miss. Susan Parker ""Suzette""",262.3750
8,"Ryerson, Miss. Emily Borie",262.3750
9,"Baxter, Mr. Quigg Edmond",247.5208


Insight: Top 10 most expensive tickets

In [17]:
duckdb.sql("""
SELECT
    Embarked,
    COUNT(*) AS passengers
FROM titanic
GROUP BY Embarked
ORDER BY passengers DESC;
""").df()

,Embarked,passengers
0,S,644
1,C,168
2,Q,77
3,None,2


Insight: Number of passengers who embarked from each port

In [18]:
duckdb.sql("""
SELECT
    COUNT(*) AS children
FROM titanic
WHERE Age < 18;
""").df()

,children
0,113


Insight: Passengers younger than 18

In [19]:
duckdb.sql("""
SELECT
    Pclass,
    COUNT(*) AS total_passengers
FROM titanic
GROUP BY Pclass
HAVING COUNT(*) > 200;
""").df()

,Pclass,total_passengers
0,1,216
1,3,491


Insight: Passenger classes with more than 200 passengers (HAVING)

In [20]:
duckdb.sql("""
SELECT
    Name,
    Age,
    CASE
        WHEN Age < 18 THEN 'Child'
        WHEN Age BETWEEN 18 AND 59 THEN 'Adult'
        ELSE 'Senior'
    END AS Age_Group
FROM titanic
LIMIT 20;
""").df()

,Name,Age,Age_Group
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult
5,"Moran, Mr. James",NaN,Senior
6,"McCarthy, Mr. Timothy J",54.0,Adult
7,"Palsson, Master. Gosta Leonard",2.0,Child
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,Adult
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,Child


Concept learned: CASE WHEN

In [21]:
duckdb.sql("""
SELECT
    CASE
        WHEN Age < 18 THEN 'Child'
        WHEN Age BETWEEN 18 AND 59 THEN 'Adult'
        ELSE 'Senior'
    END AS Age_Group,
    COUNT(*) AS Total
FROM titanic
GROUP BY Age_Group
ORDER BY Total DESC;
""").df()

,Age_Group,Total
0,Adult,575
1,Senior,203
2,Child,113


Insight: Count passengers in each age group

In [22]:
duckdb.sql("""
SELECT
    ROUND(
        100.0 * SUM(Survived) / COUNT(*),
        2
    ) AS Survival_Percentage
FROM titanic;
""").df()

,Survival_Percentage
0,38.38


Concept learned: Percentage calculation 

In [23]:
duckdb.sql("""
SELECT
    Name,
    Fare,
    RANK() OVER (ORDER BY Fare DESC) AS Fare_Rank
FROM titanic
LIMIT 10;
""").df()

,Name,Fare,Fare_Rank
0,"Ward, Miss. Anna",512.3292,1
1,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1
2,"Lesurer, Mr. Gustave J",512.3292,1
3,"Fortune, Mr. Charles Alexander",263.0000,4
4,"Fortune, Miss. Mabel Helen",263.0000,4
5,"Fortune, Miss. Alice Elizabeth",263.0000,4
6,"Fortune, Mr. Mark",263.0000,4
7,"Ryerson, Miss. Emily Borie",262.3750,8
8,"Ryerson, Miss. Susan Parker ""Suzette""",262.3750,8
9,"Baxter, Mr. Quigg Edmond",247.5208,10


Concept learned: Window functions ((RANK))